In [0]:
import os
import pandas as pd

from multiprocessing.pool import ThreadPool
from functools import partial

In [0]:
def get_files_names_list(schema_name):
   """
   Gets a list of file names from a specified directory.

   Args:
      schema_name: The name of the directory (schema) to list files from.

   Returns:
      A list of cleaned file names without the .csv extension.
   """

   directory_path = f"./{schema_name}"

   files_list = os.listdir(directory_path)

   cleaned_files_list = []

   for file_name in files_list:
      file_name = file_name.removesuffix(".csv")
      cleaned_files_list.append(file_name)

   return cleaned_files_list

In [0]:
def write_db_csvs_to_delta(schema_name, file_name):
    """
    Reads a CSV, converts it to a Spark DataFrame, and writes it as a Delta table.

    Args:
        schema_name: The schema/directory containing the CSV file.
        file_name: The name of the file to be processed (without the .csv extension).
    """
    csv_path = f"./{schema_name}/{file_name}.csv"
    
    # In Databricks Community Edition, we don't have direct filesystem access 
    # to read uploaded files with spark.read.csv. 
    # We use pandas as a workaround to read the local file into a DataFrame first.
    pandas_df = pd.read_csv(csv_path)
   
    spark_df = spark.createDataFrame(pandas_df)
    
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(file_name)

In [0]:
def process_files_in_parallel_pandas(files_list, schema_name):
    """
    Processes a list of files in parallel using a thread pool.

    Uses functools.partial to pass the schema_name argument to the mapping function.

    Args:
        files_list: A list of file names to process.
        schema_name: The schema/directory where the files are located.
    """
    num_threads = 8
    
    with ThreadPool(num_threads) as pool:
        task = partial(write_db_csvs_to_delta, schema_name)
        pool.map(task, files_list)

In [0]:
catalog_name="fea_academy"
folder_names=["adventure_works", "northwind"]

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

for folder in folder_names:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{folder}")
    spark.sql(f"USE SCHEMA {folder}")

    files_list = get_files_names_list(folder)

    process_files_in_parallel_pandas(files_list, folder)

    print(f"Finished processing {folder} files")